# 05 — Two servers, one agent

Each science domain lives in its own MCP server —
[`spectra-mcp-server`](https://github.com/HEP-KE/spectra-mcp-server)
(cosmology) and
[`gaia-mcp-server`](https://github.com/HEP-KE/gaia-mcp-server) (stellar
astrophysics) — maintained, tested, and versioned independently, like
microservices for science. The client doesn't care: `MultiServerMCPClient`
takes a dict of servers, and the agent sees one flat tool list.

This notebook connects to **both at once** and hands the agent a task that
needs tools from each.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

OUTPUT_DIR = str((Path.cwd().parent / "agent-output").resolve())
SPECTRA_REPO = str((Path.cwd().parent.parent / "spectra-mcp-server").resolve())
GAIA_REPO = str((Path.cwd().parent.parent / "gaia-mcp-server").resolve())

from agents import build_graph, load_tools, make_llm, new_run

MULTI_CONFIG = {
    "spectra": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["-m", "mcp_server", "--transport", "stdio"],
        "cwd": SPECTRA_REPO,
    },
    "gaia": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["-m", "mcp_server", "--transport", "stdio"],
        "cwd": GAIA_REPO,
    },
}

tools = await load_tools(MULTI_CONFIG)
print(f"{len(tools)} tools from two servers:\n")
for t in tools:
    print(f"  {t.name:28s} {t.description.strip().splitlines()[0][:58]}")

Over HTTP it is the same idea with two ports — start each server in its own
terminal and point the config at both:

```python
MULTI_CONFIG = {
    "spectra": {"transport": "streamable_http", "url": "http://127.0.0.1:8000/mcp"},
    "gaia":    {"transport": "streamable_http", "url": "http://127.0.0.1:8001/mcp"},
}
```

And this composability is not specific to our client: the same two servers
plug into Claude Code, the Claude desktop app, Codex, or Cursor
side-by-side — ready-made configs are in each server repo's
`docs/mcp-clients.md`.

In [ ]:
llm = make_llm("groq")
graph = build_graph(llm, tools)

## One task, two sciences

In [ ]:
TASK = f"""Produce two results and save every file to {OUTPUT_DIR}:

1. Cosmology: compute the linear matter power spectrum at z=0 for standard
   LCDM and for LCDM with total neutrino mass 0.15 eV, and plot both against
   the eBOSS DR14 Lyman-alpha forest data with LCDM as the ratio reference.

2. Stars: build the Gaia DR2 100 pc Hertzsprung-Russell diagram — fetch the
   sample with source="bundled", apply the published quality filters, compute
   absolute magnitudes, and draw the density colour-magnitude diagram.

In the final report give the neutrino suppression you observe at
k = 1 h/Mpc and the filtered Gaia star count vs the published 212,728."""

initial_state = new_run(TASK)

async for update in graph.astream(initial_state, stream_mode="updates"):
    for node, delta in update.items():
        print(f"=== {node} " + "=" * (60 - len(node)))
        if "plan" in delta:
            for step in delta["plan"]:
                print(f"  plan {step['id']}: {step['description']}")
        elif "final_report" in delta:
            final_report = delta["final_report"]
            print("  report ready")
        else:
            print(f"  {delta['step_results'][-1]}")

In [ ]:
from IPython.display import Markdown

Markdown(final_report)

In [ ]:
from IPython.display import Image, display

display(Image(f"{OUTPUT_DIR}/power_spectrum_comparison.png", width=560))
display(Image(f"{OUTPUT_DIR}/gaia_cmd_hrd.png", width=520))

## What this shows

- **Composition without integration work.** Neither server knows the other
  exists; no client code changed. The agent routed cosmology steps to CLASS
  tools and stellar steps to Gaia tools because the tool schemas told it
  what does what.
- **Tool names are the only namespace.** The two servers' tool names don't
  collide; if they ever did, prefix them server-side (`spectra_plot`,
  `gaia_plot`) — schemas are the contract.
- **This is how it scales.** A third domain is a third repo with a `tools/`
  package and one more entry in the config dict — the pattern the whole
  tutorial has been building toward.